In [45]:
import os
from openimages.download import download_dataset

In [46]:
data_dir = 'data'
number_of_samples = 350
classes = ['Missile', 'Balloon', 'Castle']

In [47]:
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

In [48]:
if not (os.path.exists(os.path.join(data_dir, classes[0].lower())) and
        os.path.exists(os.path.join(data_dir, classes[1].lower())) and
        os.path.exists(os.path.join(data_dir, classes[2].lower()))):
    download_dataset(data_dir, classes, limit=number_of_samples)

In [49]:
import torch
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import numpy as np
from skimage import io
from skimage.transform import resize
from skimage.color import gray2rgb
import glob

In [50]:
def read_img(file_name):
    img = io.imread(file_name)
    if img.ndim == 2:
        img = gray2rgb(img)
    img = resize(img, (224, 224))
    img = torch.tensor(img)
    img = img.permute(2, 0, 1)
    return img.float()

In [51]:
class CustomDataset(Dataset):
    def __init__(self, images_dir):
        self.images_dir = images_dir
        self.transforms = transforms

        self.class1_files = glob.glob(self.images_dir + '/{}/images/*.jpg'.format(classes[0].lower()))
        self.class2_files = glob.glob(self.images_dir + '/{}/images/*.jpg'.format(classes[1].lower()))
        self.class3_files = glob.glob(self.images_dir + '/{}/images/*.jpg'.format(classes[2].lower()))
        self.class1_len = len(self.class1_files)
        self.class2_len = len(self.class2_files)
        self.class3_len = len(self.class3_files)

        self.files = self.class1_files + self.class2_files + self.class3_files

        self.labels = np.zeros(len(self.files))
        self.labels[self.class1_len:self.class1_len + self.class2_len] = 1
        self.labels[self.class1_len + self.class2_len:] = 2

        self.order = [x for x in np.random.permutation(len(self.labels))]
        self.files = [self.files[x] for x in self.order]
        self.labels = [self.labels[x] for x in self.order]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        file = self.files[i]
        im = read_img(file)

        img = im.clone().detach()

        y = self.labels[i]
        return img, y

In [52]:
dataset = CustomDataset(data_dir)
train_dataset = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=4, prefetch_factor=2)
iterator = iter(train_dataset)

In [53]:
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)
model.eval()
pass

In [54]:
class_indices = []
for cls in classes:
    idx = weights.meta['categories'].index(cls.lower())
    class_indices.append(idx)

In [55]:
def prepare_batch(images):
    batch = []
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    for img in images:
        batch.append((img - mean) / std)

    return torch.cat(batch, dim=0)

In [56]:
tp = [0, 0, 0]
fp = [0, 0, 0]
tn = [0, 0, 0]
fn = [0, 0, 0]

all_scores = []
all_labels = []

with torch.no_grad():
    for images, labels in iterator:
        batch = prepare_batch(images)
        predictions = model(batch).softmax(1)

        all_scores.append(predictions[:, class_indices])
        all_labels.append(labels)

all_scores = torch.cat(all_scores).numpy()
all_labels = torch.cat(all_labels).numpy()

In [57]:
tp = np.zeros(3)
fp = np.zeros(3)
tn = np.zeros(3)
fn = np.zeros(3)

threshold = 0.09
for c in range(3):
    predicted_positive = all_scores[:, c] >= threshold
    actual_positive = all_labels == c

    tp[c] = (predicted_positive & actual_positive).sum()
    fp[c] = (predicted_positive & ~actual_positive).sum()
    tn[c] = (~predicted_positive & ~actual_positive).sum()
    fn[c] = (~predicted_positive & actual_positive).sum()

accuracy = (tp + tn) / (tp + fp + tn + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print('Classes: ', classes)
print('True positive: ', tp)
print('False positive: ', fp)
print('True negative: ', tn)
print('False negative: ', fn)

print()
print('Classes: ', classes)
print("Accuracy: ", accuracy * 100)
print("Precision: ", precision * 100)
print("Recall: ", recall * 100)
print("F1: ", f1 * 100)

print()
print('Classes: ', classes)
print('Mean accuracy: ', accuracy.mean() * 100)
print('Mean precision: ', precision.mean() * 100)
print('Mean recall: ', recall.mean() * 100)
print('Mean F1: ', f1.mean() * 100)


Classes:  ['Missile', 'Balloon', 'Castle']
True positive:  [296. 198. 204.]
False positive:  [0. 0. 0.]
True negative:  [607. 607. 700.]
False negative:  [ 54. 152.  53.]

Classes:  ['Missile', 'Balloon', 'Castle']
Accuracy:  [94.35736677 84.11703239 94.46185998]
Precision:  [100. 100. 100.]
Recall:  [84.57142857 56.57142857 79.37743191]
F1:  [91.64086687 72.26277372 88.5032538 ]

Classes:  ['Missile', 'Balloon', 'Castle']
Mean accuracy:  90.97875304771857
Mean precision:  100.0
Mean recall:  73.50676301649064
Mean F1:  84.1356314639294
